# Sensor health screening

Everything so far has assumed the instruments were working. This notebook asks what
happens when they are not, and whether the pipeline can tell.

The question matters because of an asymmetry in how sensor faults present. Gating
already rejects readings that are obviously unusable — an implausible mixture, a sample
during overrun fuel cut. But a sensor that is *biased*, *stuck* or *sluggish* produces
entirely plausible readings that happen to be wrong. Every gating criterion accepts
them, the correction faithfully compensates for the instrument fault by bending the VE
table, and the resulting table looks exactly as trustworthy as a correct one.

That last point is the whole motivation. A table produced from faulty data carries no
mark distinguishing it from a good one, and its output sets fuelling on a real engine.

Five questions:

1. **A matched pair** — the same drive, logged by healthy and by faulty instruments.
2. **What screening reports** — the two logs side by side.
3. **The whole catalogue** — does each fault get caught, and is a healthy log left alone?
4. **Does detection track damage?** — do the faults flagged loudest hurt the table most?
5. **The blind spot** — one fault that passes screening and does real harm.

Runtime is roughly three to four minutes, almost all of it in section 4.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from vetuner.correction import (
    apply_correction,
    sample_correction_factors,
    select_samples,
)
from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.faults import FAULT_CASES, fault_case
from vetuner.gating import gate_samples
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensor_health import SensorFaultError, assert_usable, screen_log
from vetuner.sensors import apply_sensor_model
from vetuner.surface_fit import fit_surface
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

TRAIN = generate_drive_cycle()
VALIDATION = generate_drive_cycle(VALIDATION_CYCLE)
BASE = ve_table(params=BASE_MAP_SURFACE)
TRUTH = ve_table(params=TRUE_SURFACE)
GAIN, PASSES, METHOD = 0.6, 5, "gaussian_process"

print(f"training cycle:   {TRAIN.time_s[-1]:.0f} s, {len(TRAIN.time_s)} samples")
print(f"validation cycle: {VALIDATION.time_s[-1]:.0f} s, {len(VALIDATION.time_s)} samples")

One helper is needed. `iterate_calibration` re-simulates the engine on every pass, so a
fault injected into a finished log would be discarded at the start of the next pass. A
broken sensor stays broken, so the fault has to be re-applied each time the log is
regenerated.

What follows mirrors the library's own loop exactly, with one line added to inject the
fault. Keeping the rest identical is what makes the comparison against a healthy run
meaningful.

In [ ]:
def calibrate(case, cycle=TRAIN, passes=PASSES, seed=0):
    """Run the calibration with a sensor fault present on every pass.

    Mirrors ``vetuner.iteration.iterate_calibration``, adding only the fault
    injection, so results are directly comparable with the unfaulted pipeline.

    Returns the corrected table, or None if the fault stopped the pipeline
    outright rather than merely degrading it.
    """
    table = np.asarray(BASE, dtype=float).copy()
    for pass_index in range(passes):
        result = simulate_engine(cycle, table)
        log = case.apply(apply_sensor_model(result, seed=seed + pass_index))
        kept = select_samples(log, gate_samples(log).accepted)
        if len(kept) == 0:
            return None
        try:
            surface = fit_surface(
                METHOD, sample_correction_factors(kept), kept.rpm, kept.map_kpa
            )
        except ValueError:
            return None
        table = apply_correction(table, surface, gain=GAIN)
    return table


control = fault_case("healthy")
print(f"healthy baseline VE RMSE: {evaluate_table(calibrate(control), TRUTH).rmse:.2f}")

---
## 1. A matched pair of datasets

Both logs come from the *same* forward simulation of the same drive cycle, so the engine
did exactly the same thing in each. They differ only in what the instruments recorded.

That is the point of pairing them. A faulty log on its own shows only that the screening
fires; it cannot show that the screening leaves good data alone. Both failure modes
matter equally — a check that flags everything is as useless as one that flags nothing.

The fault injected here is a manifold pressure sensor reading 6 kPa high. It is the
hardest case in the set: pressure bias is the single largest sensitivity in the whole
pipeline (notebook 08), and nothing local marks the readings as wrong.

In [ ]:
sim = simulate_engine(TRAIN, BASE)

healthy = apply_sensor_model(sim, seed=0)
faulty = fault_case("pressure_bias_large").apply(healthy)

frame = pd.DataFrame({
    "healthy": [healthy.map_kpa.min(), healthy.map_kpa.mean(), healthy.map_kpa.max()],
    "faulty": [faulty.map_kpa.min(), faulty.map_kpa.mean(), faulty.map_kpa.max()],
}, index=["min", "mean", "max"]).round(2)
frame["difference"] = (frame["faulty"] - frame["healthy"]).round(2)

print("manifold pressure, kPa absolute")
print(frame.to_string())
print()
print("identical on every other channel:",
      all(np.array_equal(getattr(healthy, c), getattr(faulty, c))
          for c in ("rpm", "iat_c", "tps", "afr", "batt_v")))

A clean 6 kPa shift and nothing else touched. Now the part that makes the fault
dangerous — how little the existing machinery notices.

In [ ]:
rows = []
for label, log in (("healthy", healthy), ("faulty", faulty)):
    report = gate_samples(log)
    kept = select_samples(log, report.accepted)
    rows.append({
        "log": label,
        "samples": len(log),
        "accepted by gating": int(report.accepted.sum()),
        "acceptance rate": f"{report.accepted.mean():.1%}",
        "readings outside sensor range": int(
            ((log.map_kpa < 0.0) | (log.map_kpa > 300.0)).sum()
        ),
        "mean kept pressure": f"{kept.map_kpa.mean():.1f} kPa",
    })

print(pd.DataFrame(rows).to_string(index=False))

Gating accepts the faulty log at very nearly the same rate, and not one reading falls
outside what the sensor could physically report. Judged sample by sample the faulty data
is indistinguishable from good data, which is exactly why a separate screening step is
needed: the fault is a property of the *whole channel*, not of any one reading in it.

---
## 2. What screening reports

`screen_log` runs thirteen checks and returns a report whose worst severity decides
whether calibration should proceed. Three tiers: pass quietly, proceed with a warning
attached, or refuse.

The refusal is deliberate, and it is the one design decision here worth defending. The
obvious alternative is to repair the data and carry on. That was rejected: repairing
requires assuming the fault's form, trusting a magnitude inferred from the same suspect
data, and it leaves no mark on the output. Reporting the fault with a quantified
estimate keeps the decision, and the record of it, with the person running the tool.

In [ ]:
for label, log in (("HEALTHY LOG", healthy), ("FAULTY LOG", faulty)):
    print("=" * 74)
    print(label)
    print("=" * 74)
    report = screen_log(log)
    print(report)
    try:
        assert_usable(report)
        print("\n-> proceeding with calibration")
    except SensorFaultError as error:
        print(f"\n-> refused: {str(error).splitlines()[0]}")
    print()

The healthy log passes all thirteen checks. The faulty one is refused, and the message
says which sensor, by how much, and on what physical grounds.

That last part is what makes the check possible at all. A naturally aspirated engine
cannot raise manifold pressure above ambient, so at wide-open throttle the reading must
approach barometric pressure without exceeding it. That gives an absolute reference
inside the log itself — no second sensor, no external calibration. A reading of 107 kPa
at full throttle is not merely unusual, it is impossible.

In [ ]:
wot = healthy.tps > 0.90
print(f"wide-open-throttle samples available: {int(wot.sum())}")
print()
for label, log in (("healthy", healthy), ("faulty", faulty)):
    observed = np.percentile(log.map_kpa[wot], 99)
    print(f"{label:<8} 99th percentile at WOT: {observed:6.1f} kPa"
          f"   ({observed - 101.3:+.1f} against ambient)")

---
## 3. The whole catalogue

One fault proves little. `vetuner.faults.FAULT_CASES` holds eleven, each pairing an
injector with the verdict screening is *expected* to reach. That turns the catalogue
into a specification rather than a demonstration: the test suite asserts these verdicts,
so a check that stops working, or starts firing on healthy data, fails the build.

The catalogue deliberately contains two cases that are not detectable faults — a healthy
control, and one real fault known to pass screening. A catalogue of only the cases that
work would overstate what the screening does.

In [ ]:
rows = []
for case in FAULT_CASES:
    report = screen_log(case.apply(healthy))
    problems = report.problems()
    rows.append({
        "fault": case.name,
        "represents": case.description,
        "expected": case.expected_severity,
        "reported": report.worst,
        "agrees": "yes" if report.worst == case.expected_severity else "NO",
        "finding": problems[0].message if problems else "-",
    })

catalogue = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 60)
print(catalogue[["fault", "expected", "reported", "agrees"]].to_string(index=False))
print()
print(f"all verdicts as declared: {(catalogue['agrees'] == 'yes').all()}")

In [ ]:
for row in catalogue.itertuples():
    print(f"{row.fault}")
    print(f"  {row.represents}")
    print(f"  [{row.reported}] {row.finding}")
    print()

Nine of the eleven cases are faults screening is meant to catch, and it catches all
nine, naming the right channel each time and quantifying the fault where it can. The
healthy control passes. The eleventh is the subject of section 5.

Worth noting how differently the nine are found. Three distinct kinds of evidence are
in play, and none of them would catch the others' faults:

- **Physical impossibility** — a reading no working sensor could produce, either outside
  its range entirely or above ambient pressure at full throttle.
- **Refusal to move** — a channel that holds one value while the engine is still
  changing. Caught by comparing the longest unchanging stretch against the fraction of a
  second a working, noisy sensor manages.
- **Structural damage to the record** — missing samples, or a time base with a hole in
  it, which breaks the uniform-interval assumption every rate calculation depends on.

---
## 4. Does detection track damage?

The obvious way to justify the screening is to show that the faults it flags loudest are
the ones that hurt most. So: run the full calibration on each faulty log and compare what
screening said against what the fault actually cost.

Table error is measured against the known truth, available only because the data is
synthetic. Held-out AFR error is measured on the validation cycle, which the calibration
never saw — the honest estimate, and the one that would transfer to a real engine.

In [ ]:
rows = []
for case in FAULT_CASES:
    table = calibrate(case)
    rows.append({
        "fault": case.name,
        "severity": screen_log(case.apply(healthy)).worst,
        "ve_rmse": np.nan if table is None else evaluate_table(table, TRUTH).rmse,
        "afr_pct": np.nan if table is None else evaluate_afr(table, VALIDATION).rmse_pct,
        "ran": table is not None,
    })

damage = pd.DataFrame(rows)
baseline = float(damage.loc[damage["fault"] == "healthy", "ve_rmse"].iloc[0])
damage["cost"] = damage["ve_rmse"] - baseline


def _fmt(value):
    return "pipeline failed" if np.isnan(value) else f"{value:.2f}"


shown = pd.DataFrame({
    "fault": damage["fault"],
    "screening": damage["severity"],
    "VE RMSE": damage["ve_rmse"].map(_fmt),
    "held-out AFR %": damage["afr_pct"].map(_fmt),
    "cost vs healthy": damage["cost"].map(lambda v: _fmt(v) if not np.isnan(v) else "-"),
})
print(shown.to_string(index=False))

In [ ]:
order = {"ok": 0, "warning": 1, "critical": 2}
colours = {"ok": "#6fc96f", "warning": "#f2a03c", "critical": "#e2564a"}

fig, ax = plt.subplots(figsize=(10, 6))
seen = set()
for severity, group in damage[damage["ran"]].groupby("severity"):
    ranked = group.sort_values("ve_rmse").reset_index(drop=True)
    for i, row in ranked.iterrows():
        # Spread points within their column and alternate the label side, so
        # the cluster sitting on the baseline stays readable.
        x = order[severity] + (i - (len(ranked) - 1) / 2) * 0.13
        ax.scatter(x, row.ve_rmse, s=95, color=colours[severity],
                   edgecolor="#2b2f36", linewidth=0.6, zorder=3,
                   label=severity if severity not in seen else None)
        seen.add(severity)
        ax.annotate(
            row.fault, (x, row.ve_rmse), textcoords="offset points",
            xytext=(0, 13 if i % 2 == 0 else -17), ha="center", fontsize=7.5,
            rotation=12,
        )

ax.axhline(baseline, color="#8b95a3", linestyle="--", linewidth=1)
ax.text(-0.42, baseline, "healthy\nbaseline", fontsize=7.5, color="#8b95a3",
        va="center", ha="center")
ax.set_xticks([0, 1, 2], ["passed", "warning", "refused"])
ax.set_xlim(-0.6, 2.5)
ax.set_ylim(0.4, damage["ve_rmse"].max() * 1.15)
ax.set_xlabel("what screening reported")
ax.set_ylabel("VE table RMSE (percentage points)")
ax.set_title("Severity reported against damage actually done — they barely correlate",
             loc="left", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend(title="severity", fontsize=8, title_fontsize=8, loc="upper left")
crashed = list(damage.loc[~damage["ran"], "fault"])
if crashed:
    ax.text(0.99, 0.02, f"not plotted, pipeline failed: {', '.join(crashed)}",
            transform=ax.transAxes, ha="right", fontsize=7.5, color="#8b95a3")
fig.tight_layout()
plt.show()

This is not the result the section set out to find, and it is more interesting than the
one it expected.

**Severity does not predict table damage.** Only two of the nine detected faults cost the
VE table anything: the biased pressure sensor (+2.34 points) and the frozen one (+7.68).
Five faults — including three refused outright — cost essentially nothing, and one does
not produce a bad table at all because it stops the pipeline instead.

That demands an explanation rather than a shrug, and there is a clean one. Two separate
mechanisms are protecting the table, and neither is the screening.

**Gating already discards the bad samples.** For any fault that produces an implausible
*value*, the criteria from notebook 03 throw those samples away before the correction
sees them.

In [ ]:
rows = []
for case in FAULT_CASES:
    log = case.apply(healthy)
    if not np.all(np.isfinite(log.afr)):
        rows.append({"fault": case.name, "accepted": "-", "change": "fit rejects NaN"})
        continue
    accepted = int(gate_samples(log).accepted.sum())
    rows.append({
        "fault": case.name,
        "accepted": accepted,
        "change": f"{accepted - 9194:+d}",
    })
print(pd.DataFrame(rows).to_string(index=False))

The saturated wideband loses 5570 samples to the implausible-mixture criterion, and the
impossible engine speeds lose 521 to the out-of-domain criterion. The remaining data is
clean, so the table survives. Gating is doing real work here — but silently. Nothing in
its output says a sensor is broken.

**The correction ratio is blind to temperature by construction.** The faults on the
temperature channel change *nothing at all*, which is worth checking rather than assuming:

In [ ]:
from vetuner.correction import sample_correction_factors

dead = fault_case("temperature_dead").apply(healthy)
kept_healthy = select_samples(healthy, gate_samples(healthy).accepted)
kept_dead = select_samples(dead, gate_samples(dead).accepted)

print("correction factors bit-for-bit identical:", np.array_equal(
    sample_correction_factors(kept_healthy), sample_correction_factors(kept_dead)
))

Identical, and necessarily so. The correction rests on

$$\frac{\text{AFR}_\text{measured}}{\text{AFR}_\text{target}}
  = \frac{\text{VE}_\text{true}}{\text{VE}_\text{table}}$$

in which cylinder volume and air density cancel. Intake temperature enters only through
density, so it cannot reach the VE correction at all. Engine speed and manifold pressure
are different: they are the table's *axes*, so a fault there puts samples in the wrong
cell. That is precisely why the two pressure faults are the only ones that damage the
table, and why a frozen pressure sensor is the worst of the set — every sample from the
moment it failed lands in one column.

So a dead temperature sensor leaves the VE table untouched. It does not leave the
*fuelling* untouched, because pulse width uses density directly:

In [ ]:
from vetuner.pulse_width import pulse_width_map

table = calibrate(control)
true_cold = pulse_width_map(table, iat_c=27.8).pulse_width_ms
stuck_warm = pulse_width_map(table, iat_c=36.75).pulse_width_ms

print(f"intake air genuinely at 27.8 C, sensor stuck at 36.8 C")
print(f"  mean pulse width commanded: {stuck_warm.mean():.3f} ms")
print(f"  mean pulse width required:  {true_cold.mean():.3f} ms")
print(f"  fuelling error: {abs(stuck_warm / true_cold - 1).mean() * 100:.2f}% lean")

Which settles what the severity tiers actually mean. They are not a prediction of VE
RMSE on this drive cycle — they report whether the instrument can be trusted. A
temperature sensor reporting one value all run is broken, and refusing is right even
though this particular cycle's narrow temperature range makes the table unharmed. On a
cold start, or a hot day, the same fault misfuels by a couple of percent.

Three distinct things the screening earns, then, and none of them is the one this section
went looking for:

- It catches the two faults that genuinely corrupt the table, which gating cannot see
  because the readings stay plausible sample by sample.
- It *names* the broken sensor in the cases gating silently absorbs. Without it you would
  ship a table calibrated on an engine with a dead temperature sensor, never know, and
  carry the fault into the pulse widths.
- It turns a crash into a diagnosis. Non-finite samples reach the surface fit, which
  rejects them, so the run dies in an exception from deep inside the fitting code.
  Screening reports the channel and the count instead.

One conservative call to note: the small pressure bias earns a warning while costing
nothing. The check reports any reading above ambient, and 2 kPa is above ambient while
being too small to matter. A warning proceeds rather than blocking, so the cost is wasted
attention rather than a wasted run.

The remaining case is the serious one.

---
## 5. The blind spot

Section 4 found that only two faults genuinely damage the table. There is a third, and
screening does not see it: a wideband sensor whose response has slowed, as happens with
age and contamination. It attacks the correction ratio itself, which is the one quantity
nothing else protects.

In [ ]:
case = fault_case("mixture_sluggish")
slow = case.apply(healthy)

print(f"screening verdict: {screen_log(slow).worst}")
print()
print("the channel still looks entirely reasonable:")
print(f"  range   {slow.afr.min():.2f} to {slow.afr.max():.2f}"
      f"   (healthy {healthy.afr.min():.2f} to {healthy.afr.max():.2f})")
print(f"  mean    {slow.afr.mean():.3f}        (healthy {healthy.afr.mean():.3f})")
print(f"  std     {slow.afr.std():.3f}        (healthy {healthy.afr.std():.3f})")
print()
slow_rmse = float(damage.loc[damage["fault"] == case.name, "ve_rmse"].iloc[0])
refused = damage[damage["severity"] == "critical"]["ve_rmse"]
print(f"VE RMSE healthy:          {baseline:.2f}")
print(f"VE RMSE sluggish sensor:  {slow_rmse:.2f}")
print(f"worse than {int((refused < slow_rmse).sum())} of the "
      f"{len(refused)} faults screening refuses outright")

Averages hide it completely — the fault only shows against a transient. This window
contains a throttle step from 36 to 100 kPa:

In [ ]:
window = (healthy.time_s > 182.0) & (healthy.time_s < 194.0)

fig, axes = plt.subplots(2, 1, figsize=(11, 6.2), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(healthy.time_s[window], healthy.afr[window],
             color="#3b6fc4", linewidth=1.0, label="healthy sensor")
axes[0].plot(slow.time_s[window], slow.afr[window],
             color="#e2564a", linewidth=1.6, label="sluggish sensor")
axes[0].set_ylabel("measured AFR")
axes[0].set_title("A sluggish sensor reads plausibly; it is only late",
                  loc="left", fontsize=11)
axes[0].legend(fontsize=8, loc="lower right")
axes[0].grid(alpha=0.3)
axes[0].annotate(
    "misses how rich the\ntransient actually went",
    xy=(185.3, 12.45), xytext=(186.6, 12.6), fontsize=8, color="#8b4a44",
    arrowprops=dict(arrowstyle="->", color="#8b4a44", linewidth=0.9),
)

axes[1].plot(healthy.time_s[window], healthy.map_kpa[window],
             color="#8b95a3", linewidth=1.1)
axes[1].set_ylabel("MAP (kPa)")
axes[1].set_xlabel("time (s)")
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

Nothing in the red trace is out of range, stuck, saturated or missing. It tracks the
engine correctly and simply arrives late — understating how rich the mixture went on the
step, then lagging the recovery. The correction attributes that lateness to the VE table
instead of to the sensor.

Detection was attempted and abandoned, which is worth recording as a result rather than
quietly omitting. The approach was to cross-correlate the AFR channel against manifold
pressure to measure its effective lag. It fails on two counts:

- **Weak sensitivity.** Quadrupling the sensor time constant moved the measured lag by
  0.06 s, against a baseline of 0.46 s that itself depends on the drive cycle.
- **Fundamental confounding.** Adding 0.1 s of exhaust transport delay shifted the same
  statistic by the same amount. A sluggish sensor and a long exhaust path are both lags;
  no statistic computed from one log separates them.

So the limitation is not a missing feature but a property of the measurement. Checking a
sensor's response needs a known step input, which belongs with the sensor on a bench,
not with a calibration run. The gain-error case is the same in kind: a wideband reading
3% high is indistinguishable from a table 3% wrong, which is why real wideband
controllers provide free-air calibration against a known reference.

---
## What this section establishes

Screening catches nine distinct instrument faults, names the sensor responsible, and
quantifies the fault where the physics allows. A healthy log passes all thirteen checks
across every noise seed and both drive cycles, so the checks are specific as well as
sensitive — a check that fired on good data would be worse than none.

The more useful finding is *why* it earns its place, which is not the reason assumed at
the outset. Severity does not predict table damage, because two other mechanisms already
absorb most instrument faults: gating discards implausible values, and the correction
ratio has density algebraically cancelled, leaving it blind to the temperature channel.
Only faults on the table's own axes — engine speed and manifold pressure — reach the
table, which is exactly where the barometric and freeze checks are aimed.

What screening adds is therefore diagnosis rather than protection. It names a broken
sensor that gating would otherwise absorb in silence, and that fault still reaches the
pulse widths even when the VE table escapes: a temperature sensor stuck 9 °C high misfuels
by 2.2% while leaving the table untouched.

Two limits are documented rather than hidden. A small pressure bias triggers a
conservative warning for no practical cost. More seriously, a sluggish wideband sensor
passes every check while causing more table error than half the faults that are refused.
That gap is a property of what a single log can reveal — a slow sensor and a long exhaust
path are both lags — not an omission to be fixed later. A pipeline claiming otherwise
would be offering a guarantee it cannot support.